In [ ]:
pip install openai


In [ ]:
pip install langchain langchain-openai

In [2]:
# Importar bibliotecas necesarias
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import os
import time

print("Bibliotecas importadas correctamente para streaming")

Bibliotecas importadas correctamente para streaming


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

import os

print("✓ Bibliotecas de memoria importadas correctamente")

✓ Bibliotecas de memoria importadas correctamente


In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import os
import time

print("Bibliotecas importadas correctamente para streaming")

Bibliotecas importadas correctamente para streaming


In [5]:
# Importar las bibliotecas necesarias
from openai import OpenAI
import os

# Verificar que tenemos las bibliotecas correctas
print("OpenAI library version:", __import__('openai').__version__)
print("Python version:", __import__('sys').version)

# Configuración del cliente OpenAI para GitHub Models
try:
    # Configurar el cliente con variables de entorno
    client = OpenAI(
        base_url=os.environ.get("GITHUB_BASE_URL"),
        api_key=os.environ.get("GITHUB_TOKEN")
    )

    # Verificar configuración (sin mostrar la API key completa por seguridad)
    print("Base URL configurada:", client.base_url)
    print("API Key configurada:", "✓" if client.api_key else "✗")

    if client.api_key:
        print("API Key preview:", client.api_key[:10] + "..." + client.api_key[-4:])
    else:
        print("⚠️  API Key no encontrada. Asegúrate de configurar GITHUB_TOKEN")

except Exception as e:
    print(f"Error en configuración: {e}")
    print("Verifica que las variables de entorno estén configuradas correctamente")

OpenAI library version: 2.30.0
Python version: 3.11.15 (main, Apr  7 2026, 02:25:39) [GCC 12.2.0]
Base URL configurada: https://models.inference.ai.azure.com
API Key configurada: ✓
API Key preview: ghp_E8lSOG...gki1


In [6]:
# Configuración del modelo con streaming habilitado
try:
    llm = ChatOpenAI(
        base_url=os.getenv("OPENAI_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4.1",
        streaming=True,  # ¡Importante: habilitar streaming!
        temperature=0.7
    )
    
    print("✓ Modelo configurado con streaming habilitado")
    print(f"Modelo: {llm.model_name}")
    print(f"Streaming: {llm.streaming}")
    
except Exception as e:
    print(f"✗ Error en configuración: {e}")
    print("Verifica las variables de entorno")

✓ Modelo configurado con streaming habilitado
Modelo: gpt-4.1
Streaming: True


In [7]:
# Ejemplo básico de streaming
def streaming_basico():
    prompt = "Cuéntame una historia corta sobre un programador que descubre la magia en el código"
    
    print("=== STREAMING EN TIEMPO REAL ===")
    print("Generando respuesta...")
    print("-" * 50)
    
    try:
        # stream() devuelve un generador de chunks
        for chunk in llm.stream([HumanMessage(content=prompt)]):
            # Imprimir cada chunk sin nueva línea
            print(chunk.content, end="", flush=True)
            time.sleep(0.01)  # Pequeña pausa para simular streaming visual
            
        print("\n" + "-" * 50)
        print("✓ Streaming completado")
        
    except Exception as e:
        print(f"✗ Error en streaming: {e}")

# Ejecutar streaming básico
streaming_basico()

=== STREAMING EN TIEMPO REAL ===
Generando respuesta...
--------------------------------------------------
Había una vez un programador llamado Julián, apasionado por resolver problemas y escribir líneas de código limpias. Trabajaba largas horas frente a su computadora, siempre buscando la lógica perfecta y la eficiencia máxima. Sin embargo, sentía que algo faltaba, como si el código solo sirviera para dar órdenes a máquinas, sin lugar para el asombro.

Una noche, mientras depuraba un extraño error en una aplicación, Julián notó que su pantalla parpadeaba con símbolos desconocidos. Pensando que era un fallo de hardware, intentó reiniciar, pero los símbolos comenzaron a moverse, formando palabras: “La magia está en el código”.

Intrigado, escribió una sencilla función de saludo. Al ejecutarla, las letras salieron flotando de la pantalla, danzando en el aire ante sus ojos. Julián parpadeó, incrédulo. Empezó a experimentar: un ciclo for hacía que pequeñas luces giraran alrededor de su esc

In [9]:
# Ejemplo básico con RunnableWithMessageHistory
llm = ChatOpenAI(
        base_url=os.getenv("OPENAI_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4.1",
        streaming=True,  # ¡Importante: habilitar streaming!
        temperature=0.7
)
# Prompt con historial + entrada
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de memorias por sesión
store = {}

def get_session_history(session_id: str):
    """Devuelve (o crea) el historial completo para la sesión."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Envolver con RunnableWithMessageHistory
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def ejemplo_buffer_memory():
    print("=== CONVERSATIONBUFFERMEMORY ===")
    print("Mantiene todo el historial de conversación\n")
    
    session_id = "demo_session"

    try:
        # Primera interacción
        print("1. Primera pregunta:")
        response1 = conversation.invoke(
            {"input": "Mi nombre es Maxipiri y trabajo en turismo"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response1.content}\n")

        # Segunda interacción
        print("2. Segunda pregunta:")
        response2 = conversation.invoke(
            {"input": "¿Cuál es mi nombre y profesión?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response2.content}\n")

        # Tercera interacción
        print("3. Tercera pregunta:")
        response3 = conversation.invoke(
            {"input": "¿cual es mi area de trabajo?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response3.content}\n")

        # Mostrar historial
        print("=== CONTENIDO DE LA MEMORIA ===")
        history = store[session_id].messages
        for i, msg in enumerate(history, 1):
            print(f"{i}. {msg.type}: {msg.content}")

    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_buffer_memory()


=== CONVERSATIONBUFFERMEMORY ===
Mantiene todo el historial de conversación

1. Primera pregunta:
Respuesta: ¡Hola, Maxipiri! Encantado de conocerte. Trabajar en turismo es fascinante, ya que permite conectar a las personas con nuevos lugares y experiencias. ¿En qué área del turismo trabajas? ¿Guías, agencias, hotelería, o algo diferente? Si necesitas ayuda con información sobre destinos, herramientas para mejorar tu trabajo, ideas para atraer clientes, o cualquier tema relacionado, ¡estoy aquí para ayudarte!

2. Segunda pregunta:
Respuesta: Tu nombre es Maxipiri y trabajas en turismo.

3. Tercera pregunta:
Respuesta: Hasta ahora, solo mencionaste que trabajas en turismo, pero no especificaste tu área exacta dentro del sector. Si quieres, cuéntame más: ¿te dedicas a guiar turistas, trabajar en agencias de viajes, hotelería, organización de eventos, o alguna otra especialidad? ¡Estoy aquí para ayudarte en lo que necesites!

=== CONTENIDO DE LA MEMORIA ===
1. human: Mi nombre es Maxipiri